# CH09 — Colas de prioridad (Priority Queues)

Material del curso basado en Goodrich, Tamassia & Goldwasser — Capítulo 9.

## 9.1 ADT — Cola de prioridad

- Una **cola de prioridad** almacena una colección de **entradas** (*entries*), cada una con una **clave** (*key*, la prioridad) y un **valor** asociado (*value*).
- A diferencia de una cola FIFO, no importa el orden de llegada: la operación de remoción siempre entrega la entrada con la **clave mínima** (convención *min-oriented*, la que usa Goodrich).
- Las claves deben admitir un **orden total** (se pueden comparar entre sí con `<`); si dos entradas empatan en clave, el desempate es arbitrario.

**Aplicaciones típicas:** la cola de espera de un vuelo (por categoría tarifaria, no por orden de llegada), el planificador de procesos de un sistema operativo (ejecuta primero el proceso de mayor prioridad), simulación de eventos discretos (siempre se procesa el evento con el timestamp más próximo), y el algoritmo de Dijkstra de caminos mínimos (siempre expande el vértice con distancia provisional más pequeña).

### ADT

| Método | Descripción |
|---|---|
| `len(P)` | Número de entradas en `P` |
| `P.is_empty()` | `True` si `P` no tiene entradas |
| `P.add(k, v)` | Inserta una entrada con clave `k` y valor `v` |
| `P.min()` | Retorna (sin eliminar) la tupla `(k, v)` con clave mínima. Lanza `Empty` si `P` está vacía |
| `P.remove_min()` | Elimina y retorna la tupla `(k, v)` con clave mínima. Lanza `Empty` si `P` está vacía |

**Nota sobre la entrada compuesta:** dentro de la implementación, cada par `(key, value)` se suele envolver en una clase liviana `_Item` (con `__slots__`, sin métodos públicos) en vez de guardar dos listas paralelas o tuplas sueltas. Eso permite definir `__lt__` sobre `_Item` (comparando solo por `_key`) y reutilizar comparaciones de Python (`<`, `sort()`, etc.) directamente sobre los objetos guardados.

## 9.2 Implementación con lista no ordenada

La idea más simple: guardar las entradas en una lista **sin ningún orden particular**.

- `add(k, v)`: se agrega al final de la lista. **$O(1)$** amortizado — no hay que buscar dónde insertar.
- `min()` / `remove_min()`: como no hay orden, hay que **recorrer toda la lista** para encontrar la clave mínima. **$O(n)$**.

Es la estructura correcta cuando se insertan muchas entradas pero rara vez se consulta el mínimo.

### Para la clase: implementa `UnsortedPriorityQueue`

`PriorityQueueBase` ya está completa — define la clase interna `_Item` (con su `__lt__`, que compara solo por `_key`) y el método `is_empty()`, que se apoya en `len(self)`.

Completa `UnsortedPriorityQueue`, que hereda de `PriorityQueueBase` y guarda las entradas en `self._data` (una lista de objetos `_Item`, sin orden):

- `_find_min()`: método auxiliar (no forma parte del ADT) que recorre `self._data` y retorna el **índice** del `_Item` con clave mínima. Lo usan tanto `min()` como `remove_min()`, así evitan duplicar la búsqueda.
- `add(key, value)`: agrega un nuevo `_Item` al final de `self._data`.
- `min()`: usa `_find_min()` para ubicar la entrada mínima y retorna `(item._key, item._value)`, **sin** eliminarla.
- `remove_min()`: usa `_find_min()`, guarda la entrada, la elimina de `self._data` (`pop(indice)`) y retorna `(item._key, item._value)`.

Completa cada método marcado con `pass`.

In [ ]:
class PriorityQueueBase:
    """Clase base abstracta para una cola de prioridad orientada al mínimo."""

    class _Item:
        """Composite ligero para guardar clave y valor de una entrada."""
        __slots__ = '_key', '_value'

        def __init__(self, k, v):
            self._key = k
            self._value = v

        def __lt__(self, other):
            return self._key < other._key   # compara entradas solo por su clave

        def __repr__(self):
            return f'({self._key},{self._value})'

    def is_empty(self):
        """Return True if the priority queue is empty."""
        return len(self) == 0

In [ ]:
class UnsortedPriorityQueue(PriorityQueueBase):
    """Cola de prioridad orientada al mínimo, con una lista sin ordenar."""

    def _find_min(self):
        """Return the index of the item with minimum key. Lanza Empty si está vacía."""
        pass

    def __init__(self):
        """Create a new empty Priority Queue."""
        self._data = []

    def __len__(self):
        """Return the number of items in the priority queue."""
        return len(self._data)

    def add(self, key, value):
        """Add a key-value pair."""
        pass

    def min(self):
        """Return but do not remove (k,v) tuple with minimum key. Lanza Empty si está vacía."""
        pass

    def remove_min(self):
        """Remove and return (k,v) tuple with minimum key. Lanza Empty si está vacía."""
        pass

In [ ]:
# --- Tests: UnsortedPriorityQueue ---
P = UnsortedPriorityQueue()
P.add(5, 'A')
P.add(9, 'C')
P.add(3, 'B')
P.add(7, 'D')

print(len(P))              # 4
print(P.min())              # (3, 'B')   -- no elimina
print(len(P))              # 4  (sigue igual)
print(P.remove_min())       # (3, 'B')
print(P.remove_min())       # (5, 'A')
print(len(P))              # 2
print(P.is_empty())         # False

## 9.3 Implementación con lista ordenada

La estrategia opuesta: mantener las entradas **siempre ordenadas** por clave, de menor a mayor.

- `min()` / `remove_min()`: la clave mínima siempre está en la **primera posición**. **$O(1)$**.
- `add(k, v)`: hay que **recorrer** la lista para encontrar dónde insertar (para no romper el orden) y desplazar elementos. **$O(n)$**.

Es la estructura correcta cuando se inserta poco pero se consulta o elimina el mínimo con mucha frecuencia — el trade-off exactamente inverso al de la lista no ordenada.

### Para la clase: implementa `SortedPriorityQueue`

De nuevo hereda de `PriorityQueueBase` y usa `self._data`, pero esta vez la lista se mantiene **siempre ordenada** (clave ascendente) después de cada `add`.

- `add(key, value)`: crea el nuevo `_Item` y lo inserta en la posición correcta recorriendo `self._data` **de atrás hacia adelante** (`walk` empieza en el último índice) mientras el nuevo ítem sea menor que `self._data[walk]`; usa `self._data.insert(indice, item)` para insertarlo en la posición final.
- `min()`: la clave mínima siempre está en `self._data[0]`. Retorna `(item._key, item._value)` sin eliminar.
- `remove_min()`: elimina y retorna `self._data.pop(0)` como `(item._key, item._value)`.

Completa cada método marcado con `pass`.

In [ ]:
class SortedPriorityQueue(PriorityQueueBase):
    """Cola de prioridad orientada al mínimo, con una lista siempre ordenada."""

    def __init__(self):
        """Create a new empty Priority Queue."""
        self._data = []

    def __len__(self):
        """Return the number of items in the priority queue."""
        return len(self._data)

    def add(self, key, value):
        """Add a key-value pair."""
        pass

    def min(self):
        """Return but do not remove (k,v) tuple with minimum key. Lanza Empty si está vacía."""
        pass

    def remove_min(self):
        """Remove and return (k,v) tuple with minimum key. Lanza Empty si está vacía."""
        pass

In [ ]:
# --- Tests: SortedPriorityQueue ---
S = SortedPriorityQueue()
S.add(5, 'A')
S.add(9, 'C')
S.add(3, 'B')
S.add(7, 'D')

print(len(S))              # 4
print(S.min())              # (3, 'B')   -- no elimina
print(S.remove_min())       # (3, 'B')
print(S.remove_min())       # (5, 'A')
print(S.remove_min())       # (7, 'D')
print(S.remove_min())       # (9, 'C')
print(S.is_empty())         # True

## 9.4 Montículos (heaps)

Ni la lista no ordenada ni la ordenada logran que **ambas** operaciones (`add` y `remove_min`) sean rápidas a la vez: una es $O(1)$ mientras la otra es $O(n)$. Un **montículo** (*heap*) logra que **las dos** sean $O(\log n)$.

Un **montículo binario** (min-heap) es un **árbol binario completo** que cumple la **propiedad de orden de montículo**:

- **Árbol binario completo:** todos los niveles están completamente llenos, excepto posiblemente el último, que se llena **de izquierda a derecha**. Esto garantiza que la altura sea siempre $O(\log n)$ — no se puede degenerar en una lista como un BST desbalanceado.
- **Propiedad de orden:** la clave de cada nodo es **mayor o igual** que la clave de su padre (equivalentemente: la raíz siempre tiene la clave mínima de todo el árbol).

**Representación con arreglo:** como el árbol es siempre completo, se puede guardar de forma compacta en una lista, **sin punteros**, numerando los nodos por nivel (level-numbering): la raíz en el índice `0`, y para un nodo en el índice `j`:

$$\text{padre}(j) = \left\lfloor \frac{j-1}{2} \right\rfloor \qquad \text{izq}(j) = 2j+1 \qquad \text{der}(j) = 2j+2$$

Esta es la misma idea de `BinaryTree` con arreglo que se vio en CH08 para árboles completos — aquí se aplica al montículo.

![Montículo como árbol y como arreglo](../assets/ch09_heap_tree_array.png)

## 9.5 `add` y `remove_min` en un montículo: up-heap y down-heap

Insertar o eliminar puede romper momentáneamente la propiedad de orden. Se restaura "burbujeando" el elemento afectado hacia arriba o hacia abajo.

**`add(k, v)` — up-heap:**
1. Se agrega el nuevo ítem en la **primera posición libre** al final del arreglo (mantiene el árbol completo).
2. Mientras el nuevo ítem sea **menor** que su padre, se **intercambia** con él (sube un nivel).
3. Se repite hasta llegar a la raíz o hasta que el padre ya sea menor o igual.

Como la altura del árbol es $O(\log n)$, el ítem sube a lo sumo $O(\log n)$ niveles: **$O(\log n)$**.

**`remove_min()` — down-heap:**
1. La clave mínima está en la raíz (índice `0`) — se guarda para retornarla.
2. Se mueve el **último** elemento del arreglo a la raíz (mantiene el árbol completo) y se elimina la última posición.
3. Mientras la nueva raíz sea **mayor** que alguno de sus hijos, se intercambia con el **hijo de menor clave** (baja un nivel).
4. Se repite hasta llegar a una hoja o hasta que ya sea menor o igual que ambos hijos.

También **$O(\log n)$**, por la misma razón: la altura del árbol.

![Up-heap y down-heap](../assets/ch09_upheap_downheap.png)

### Para la clase: implementa `HeapPriorityQueue`

Los índices auxiliares (`_parent`, `_left`, `_right`, `_has_left`, `_has_right`, `_swap`) ya están implementados — son puramente aritmética de índices sobre `self._data`.

Completa la parte algorítmica:

- `_upheap(j)`: si `j` no es la raíz (`j > 0`) y `self._data[j]` es menor que su padre, los intercambia (`self._swap`) y **se llama a sí mismo recursivamente** sobre la posición del padre (`self._parent(j)`).
- `_downheap(j)`: si `j` tiene hijo izquierdo, determina cuál de los dos hijos (izquierdo o, si existe, derecho) tiene la clave menor (`small_child`); si ese hijo es menor que `self._data[j]`, los intercambia y **se llama a sí mismo recursivamente** sobre `small_child`.
- `add(key, value)`: agrega el nuevo `_Item` al final de `self._data` y llama a `_upheap` sobre esa posición.
- `min()`: retorna `(key, value)` del ítem en la raíz (índice `0`), sin eliminar. Lanza `Empty` si está vacía.
- `remove_min()`: intercambia la raíz con el último elemento (`self._swap(0, len(self._data) - 1)`), hace `pop()` para sacar el mínimo, llama a `_downheap(0)` para reparar la nueva raíz, y retorna `(key, value)` del ítem sacado. Lanza `Empty` si está vacía.

Completa cada método marcado con `pass`.

In [ ]:
class HeapPriorityQueue(PriorityQueueBase):
    """Cola de prioridad orientada al mínimo, implementada con un montículo binario."""

    # --- índices (ya implementados) ---
    def _parent(self, j):
        return (j - 1) // 2

    def _left(self, j):
        return 2 * j + 1

    def _right(self, j):
        return 2 * j + 2

    def _has_left(self, j):
        return self._left(j) < len(self._data)

    def _has_right(self, j):
        return self._right(j) < len(self._data)

    def _swap(self, i, j):
        """Swap the elements at indices i and j of array."""
        self._data[i], self._data[j] = self._data[j], self._data[i]

    # --- reparación del orden de montículo ---
    def _upheap(self, j):
        pass

    def _downheap(self, j):
        pass

    # --- ADT ---
    def __init__(self):
        """Create a new empty Priority Queue."""
        self._data = []

    def __len__(self):
        """Return the number of items in the priority queue."""
        return len(self._data)

    def add(self, key, value):
        """Add a key-value pair to the priority queue."""
        pass

    def min(self):
        """Return but do not remove (k,v) tuple with minimum key. Lanza Empty si está vacía."""
        pass

    def remove_min(self):
        """Remove and return (k,v) tuple with minimum key. Lanza Empty si está vacía."""
        pass

In [ ]:
# --- Tests: HeapPriorityQueue, mismo ejemplo de las figuras [4,5,6,15,9] ---
H = HeapPriorityQueue()
for k in [4, 5, 6, 15, 9]:
    H.add(k, chr(ord('A') + k))

print(len(H))               # 5
print(H.min())               # (4, 'E')   -- raíz = clave mínima

H.add(2, 'X')                 # dispara up-heap: 2 sube hasta la raíz
print(H.min())               # (2, 'X')

print(H.remove_min())        # (2, 'X')   -- dispara down-heap
print(H.remove_min())        # (4, 'E')
print(H.remove_min())        # (5, 'F')
print(len(H))               # 3

## 9.6 Comparación de complejidades

| Implementación | `add` | `min` | `remove_min` |
|---|---|---|---|
| Lista no ordenada | $O(1)$ | $O(n)$ | $O(n)$ |
| Lista ordenada | $O(n)$ | $O(1)$ | $O(1)$ |
| Montículo (heap) | $O(\log n)$ | $O(1)$ | $O(\log n)$ |

El montículo es el único que logra que **ninguna** operación sea $O(n)$ — es el punto intermedio que conviene cuando se insertan y remueven entradas con frecuencia similar.

**Heapsort:** insertar los $n$ elementos de una colección uno por uno (`add`, $O(\log n)$ cada uno) y luego extraerlos con `remove_min` repetidamente ($O(\log n)$ cada uno) produce la colección **ordenada** en $O(n \log n)$ total — es exactamente el algoritmo *heapsort*.

## 9.7 Colas de prioridad adaptables

Las operaciones anteriores solo permiten actuar sobre la entrada **mínima**. Pero en algoritmos como **Dijkstra** (caminos mínimos) se necesita, además:

- **Actualizar** la clave de una entrada que ya está en la cola (*decrease-key*: bajar la distancia provisional de un vértice cuando se encuentra un camino mejor).
- **Eliminar** una entrada arbitraria, no solo la mínima.

El problema: si solo se tiene la clave o el valor, **encontrar** esa entrada dentro del montículo cuesta $O(n)$ (recorrerlo entero) — arruinaría el $O(\log n)$ que tanto costó lograr.

**Solución: Locator.** Cada vez que se agrega una entrada, `add()` retorna un **token** (`Locator`) que el usuario guarda. El `Locator` es el mismo `_Item`, pero extendido con un campo extra `_index` que guarda su posición **actual** dentro de `self._data`. Cada vez que `_swap` intercambia dos posiciones del arreglo, también actualiza el `_index` de ambos locators — así el locator **siempre sabe dónde está** su entrada, incluso después de muchos up-heaps y down-heaps.

Con eso, `update(loc, ...)` y `remove(loc)` acceden directamente por `loc._index` en $O(1)$ (más el reacomodo de $O(\log n)$ para restaurar el orden).

![Colas de prioridad adaptables: locators](../assets/ch09_adaptable_locators.png)

### Para la clase: implementa `AdaptableHeapPriorityQueue`

Hereda de `HeapPriorityQueue` (reutiliza `_parent`, `_left`, `_right`, `_upheap`, `_downheap`, `min`, `__len__`, ...).

`Locator` ya está definida — es un `_Item` con un campo extra `_index`. `_swap` también está dado: reutiliza el `_swap` del padre (que intercambia posiciones en la lista) y **además** actualiza el `_index` de ambos locators para que reflejen su nueva posición.

Completa:

- `_bubble(j)`: método auxiliar que decide qué reparación hace falta después de un `update`. Si `j` no es la raíz y `self._data[j]` es menor que su padre, llama a `self._upheap(j)`; si no, llama a `self._downheap(j)` (cubre tanto el caso "subió de prioridad" como "bajó de prioridad").
- `add(key, value)`: crea un `Locator(key, value, len(self._data))` (con el índice donde va a quedar), lo agrega a `self._data`, hace `_upheap` sobre esa posición, y **retorna el locator** (el usuario lo necesita para poder llamar `update`/`remove` después).
- `update(loc, newkey, newval)`: valida que `loc` siga siendo válido (`0 <= loc._index < len(self)` y `self._data[loc._index] is loc`; si no, `raise ValueError('Invalid locator')`), actualiza `loc._key` y `loc._value`, y llama a `self._bubble(loc._index)` para restaurar el orden.
- `remove(loc)`: valida `loc` igual que `update`. Si `loc` está en la última posición, simplemente hace `pop()`. Si no, lo intercambia (`_swap`) con la última posición, hace `pop()`, y llama a `self._bubble(j)` sobre la posición `j` que quedó ocupada por el elemento que vino del final (pudo quedar mayor o menor que su nuevo entorno). Retorna `(loc._key, loc._value)`.

Completa cada método marcado con `pass`.

In [ ]:
class AdaptableHeapPriorityQueue(HeapPriorityQueue):
    """Cola de prioridad adaptable (con locators), implementada con un montículo."""

    class Locator(HeapPriorityQueue._Item):
        """Token que ubica una entrada dentro de la cola de prioridad."""
        __slots__ = '_index'

        def __init__(self, k, v, j):
            super().__init__(k, v)
            self._index = j

    # --- override: mantener sincronizado el índice de cada locator ---
    def _swap(self, i, j):
        super()._swap(i, j)
        self._data[i]._index = i
        self._data[j]._index = j

    def _bubble(self, j):
        pass

    def add(self, key, value):
        """Add a key-value pair. Retorna el Locator de la nueva entrada."""
        pass

    def update(self, loc, newkey, newval):
        """Update the key and value for the entry identified by Locator loc."""
        pass

    def remove(self, loc):
        """Remove and return the (k,v) pair identified by Locator loc."""
        pass

In [ ]:
# --- Tests: AdaptableHeapPriorityQueue ---
AP = AdaptableHeapPriorityQueue()
loc_a = AP.add(5, 'a')
loc_b = AP.add(9, 'b')
loc_c = AP.add(4, 'c')
loc_d = AP.add(7, 'd')

print(AP.min())                     # (4, 'c')

AP.update(loc_b, 1, 'b')             # 'b' baja su clave de 9 a 1 -> nuevo mínimo
print(AP.min())                     # (1, 'b')

print(AP.remove(loc_c))              # (4, 'c')  -- elimina una entrada que NO era el mínimo
print(len(AP))                      # 3
print(AP.min())                     # (1, 'b')  -- sigue siendo la mínima